In [1]:
import time
import requests
import chromadb
from sentence_transformers import SentenceTransformer
import os
# ====================== 全局配置 ======================
# 轻量嵌入模型，4060 12G 完美运行
minilm = os.path.join("/home/dake/Documents/Project/Life/data/datasets/all-MiniLM-L6-v2")
EMBED_MODEL = SentenceTransformer(minilm)
# Chroma 本地向量库
CHROMA_CLIENT = chromadb.PersistentClient(path="./bili_video_db")
COLLECTION = CHROMA_CLIENT.get_or_create_collection(name="bilibili_videos")
# B站请求头（防反爬）
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Referer": "https://www.bilibili.com/"
}

# ====================== 1. B站视频爬虫 ======================
def crawl_bilibili_videos(keyword: str, max_count: int = 20):
    """
    爬取B站关键词搜索的视频信息
    :param keyword: 搜索关键词
    :param max_count: 最大爬取数量
    :return: 视频列表 [{"title", "tags", "url", "bvid"}]
    """
    video_list = []
    page = 1

    while len(video_list) < max_count:
        try:
            # B站公开搜索API（无需登录）
            url = f"https://api.bilibili.com/x/web-interface/search/all/v2?keyword={keyword}&page={page}"
            resp = requests.get(url, headers=HEADERS, timeout=10)
            resp.raise_for_status()
            data = resp.json()

            # 提取视频数据
            result_items = data["data"]["result"][11]["data"]  # 11是视频类型

            for item in result_items:
                if len(video_list) >= max_count:
                    break

                # 提取字段
                title = item.get("title", "").replace("<em class=\"keyword\">", "").replace("</em>", "")
                bvid = item.get("bvid", "")
                video_url = f"https://www.bilibili.com/video/{bvid}"
                tags = item.get("tag", "").split(",")  # 标签

                video_list.append({
                    "title": title,
                    "tags": tags,
                    "url": video_url,
                    "bvid": bvid
                })

            page += 1
            time.sleep(1)  # 防爬延时

        except Exception as e:
            print(f"爬取失败：{str(e)}")
            break

    return video_list

# ====================== 2. 存入向量库 + AI分类 ======================
def save_to_vector_db(video_list: list):
    """将视频信息向量化存入Chroma，自动完成AI语义分类"""
    ids = []
    documents = []
    metadatas = []
    embeddings = []

    for idx, video in enumerate(video_list):
        # 拼接标题+标签作为语义内容
        content = f"标题：{video['title']} 标签：{','.join(video['tags'])}"
        # 生成向量
        vec = EMBED_MODEL.encode(content).tolist()

        ids.append(video["bvid"])
        documents.append(content)
        embeddings.append(vec)
        metadatas.append({
            "title": video["title"],
            "url": video["url"],
            "tags": ",".join(video["tags"])
        })

    # 批量存入向量库
    COLLECTION.add(
        ids=ids,
        documents=documents,
        embeddings=embeddings,
        metadatas=metadatas
    )
    print(f"✅ 成功存入 {len(video_list)} 个视频到向量库")

# ====================== 3. 智能Agent：检索匹配视频 ======================
def video_agent(user_query: str, top_k: int = 3):
    """
    智能Agent：根据用户输入，语义匹配视频
    :param user_query: 用户想看的内容
    :param top_k: 返回几个视频
    :return: 视频结果
    """
    print(f"\n🤖 Agent正在检索：{user_query}")
    # 用户问题向量化
    query_vec = EMBED_MODEL.encode(user_query).tolist()

    # 向量相似度检索
    results = COLLECTION.query(
        query_embeddings=query_vec,
        n_results=top_k,
        include=["metadatas", "distances"]
    )

    # 格式化输出
    output = []
    for i, meta in enumerate(results["metadatas"][0]):
        dist = round(results["distances"][0][i], 2)
        output.append({
            "排名": i+1,
            "标题": meta["title"],
            "链接": meta["url"],
            "匹配度": 100 - dist  # 距离越小越相似
        })

    return output

# ====================== 主程序 ======================
if __name__ == "__main__":
    print("="*50)
    print(" B站视频爬取 + AI分类 + 智能Agent检索 ")
    print("="*50)

    # 1. 先爬取数据（可修改关键词）
    search_key = input("请输入要爬取的B站关键词：")
    videos = crawl_bilibili_videos(keyword=search_key, max_count=20)
    save_to_vector_db(videos)

    # 2. 启动Agent循环检索
    print("\n🚀 智能Agent已启动，输入'quit'退出")
    while True:
        user_input = input("\n你想看什么类型的视频：")
        if user_input.lower() == "quit":
            print("👋 退出程序")
            break

        # Agent检索
        res = video_agent(user_input)
        for item in res:
            print(f"\n【{item['排名']}】匹配度：{item['匹配度']}%")
            print(f"标题：{item['标题']}")
            print(f"链接：{item['链接']}")


E0000 00:00:1775302879.329439  180029 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1775302879.329477  180029 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_rejected' registered more than once. Ignoring later registration.
E0000 00:00:1775302879.329481  180029 instrument.cc:563] Metric with name 'grpc.resource_quota.connections_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1775302879.329483  180029 instrument.cc:563] Metric with name 'grpc.resource_quota.instantaneous_memory_pressure' registered more than once. Ignoring later registration.
E0000 00:00:1775302879.329487  180029 instrument.cc:563] Metric with name 'grpc.resource_quota.memory_pressure_control_value' registered more than once. Ignoring later registration.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: /home/dake/Documents/Project/Life/data/datasets/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 B站视频爬取 + AI分类 + 智能Agent检索 


请输入要爬取的B站关键词： 潜鸟叫声


...


你想看什么类型的视频： 哈基米



🤖 Agent正在检索：哈基米

【1】匹配度：99.02%
标题：孔雀的叫声
链接：https://www.bilibili.com/video/BV1kY4y1t7S2

【2】匹配度：99.0%
标题：乌鸫--叫声
链接：https://www.bilibili.com/video/BV1u54y157ri

【3】匹配度：98.96%
标题：游隼的叫声
链接：https://www.bilibili.com/video/BV1XY411A73W



你想看什么类型的视频： 音乐



🤖 Agent正在检索：音乐

【1】匹配度：98.98%
标题：孔雀的叫声
链接：https://www.bilibili.com/video/BV1kY4y1t7S2

【2】匹配度：98.96%
标题：乌鸫--叫声
链接：https://www.bilibili.com/video/BV1u54y157ri

【3】匹配度：98.94%
标题：室外寻鸟必备音频｜大声播放这段视频鸟会自己飞下来
链接：https://www.bilibili.com/video/BV1PdK5eWEZZ


KeyboardInterrupt: Interrupted by user

In [4]:
import time
import requests
import chromadb
import gradio as gr
import hashlib
import json
from sentence_transformers import SentenceTransformer
import os
from openai import OpenAI
from typing import List, Dict, Any, Optional
from datetime import datetime
import pandas as pd

# ====================== 全局配置 ======================
# 轻量嵌入模型
minilm = os.path.join("/home/dake/Documents/Project/Life/data/datasets/all-MiniLM-L6-v2")
EMBED_MODEL = SentenceTransformer(minilm)

# Chroma 本地向量库
CHROMA_CLIENT = chromadb.PersistentClient(path="./bili_video_db")
COLLECTION = CHROMA_CLIENT.get_or_create_collection(name="bilibili_videos")

# 千问API客户端
QIANWEN_CLIENT = OpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url="https://dashscope.aliyuncs.com/api/v2/apps/protocols/compatible-mode/v1",
)
MODEL_NAME = "qwen-plus"

# B站请求头
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Referer": "https://www.bilibili.com/"
}

# 历史记录文件
HISTORY_FILE = "search_history.json"

# ====================== 1. 增强的B站视频爬虫 ======================
def crawl_bilibili_videos(keyword: str, max_count: int = 20) -> List[Dict[str, Any]]:
    """增强的B站视频爬虫，支持更多字段"""
    video_list = []
    page = 1
    
    print(f"🔍 开始爬取关键词: {keyword}")
    
    while len(video_list) < max_count:
        try:
            url = f"https://api.bilibili.com/x/web-interface/search/all/v2?keyword={keyword}&page={page}"
            resp = requests.get(url, headers=HEADERS, timeout=15)
            resp.raise_for_status()
            data = resp.json()
            
            # 检查是否有视频结果
            if "data" not in data or "result" not in data["data"]:
                print("⚠️ 没有更多搜索结果")
                break
                
            # 查找视频类型的结果（索引11通常是视频）
            video_results = None
            for result in data["data"]["result"]:
                if result.get("result_type") == "video":
                    video_results = result.get("data", [])
                    break
            
            if not video_results:
                print("⚠️ 没有找到视频结果")
                break
                
            for item in video_results:
                if len(video_list) >= max_count:
                    break
                    
                # 提取详细信息
                title = item.get("title", "").replace("<em class=\"keyword\">", "").replace("</em>", "")
                bvid = item.get("bvid", "")
                author = item.get("author", "")
                play = item.get("play", 0)  # 播放量
                duration = item.get("duration", "")  # 时长
                pubdate = item.get("pubdate", 0)  # 发布时间戳
                
                # 格式化发布时间
                publish_time = datetime.fromtimestamp(pubdate).strftime("%Y-%m-%d %H:%M") if pubdate else "未知"
                
                video_url = f"https://www.bilibili.com/video/{bvid}"
                
                # 获取标签
                tags = item.get("tag", "")
                if tags:
                    tag_list = [tag.strip() for tag in tags.split(",") if tag.strip()]
                else:
                    tag_list = []
                    
                # 获取描述
                description = item.get("description", "")[:200]  # 截断描述
                
                video_list.append({
                    "title": title,
                    "author": author,
                    "bvid": bvid,
                    "url": video_url,
                    "play_count": play,
                    "duration": duration,
                    "publish_time": publish_time,
                    "tags": tag_list,
                    "description": description,
                    "search_keyword": keyword,  # 记录搜索关键词
                    "crawl_time": datetime.now().isoformat()
                })
                
            print(f"📄 第{page}页爬取完成，已获取{len(video_list)}个视频")
            page += 1
            time.sleep(1.5)  # 防爬延时
            
        except requests.exceptions.Timeout:
            print("⏰ 请求超时，跳过当前页")
            break
        except Exception as e:
            print(f"❌ 爬取失败：{str(e)}")
            break
    
    print(f"✅ 爬取完成，共获取 {len(video_list)} 个视频")
    return video_list

# ====================== 2. 智能向量化存储 ======================
def save_to_vector_db(video_list: List[Dict[str, Any]]):
    """智能向量化存储，使用优化内容"""
    if not video_list:
        print("⚠️ 没有视频需要存储")
        return
    
    ids = []
    documents = []
    metadatas = []
    embeddings = []
    
    for video in video_list:
        # 优化内容组合，提高检索质量
        content_parts = [
            f"标题: {video['title']}",
            f"UP主: {video['author']}",
            f"标签: {', '.join(video['tags'])}",
            f"描述: {video['description']}",
            f"关键词: {video['search_keyword']}"
        ]
        content = " | ".join(content_parts)
        
        # 生成向量
        vec = EMBED_MODEL.encode(content).tolist()
        
        # 使用bvid + 哈希值作为唯一ID
        unique_id = f"{video['bvid']}_{hashlib.md5(content.encode()).hexdigest()[:8]}"
        
        ids.append(unique_id)
        documents.append(content)
        embeddings.append(vec)
        metadatas.append({
            "title": video["title"],
            "author": video["author"],
            "bvid": video["bvid"],
            "url": video["url"],
            "play_count": video["play_count"],
            "duration": video["duration"],
            "publish_time": video["publish_time"],
            "tags": ",".join(video["tags"]),
            "description": video["description"],
            "search_keyword": video["search_keyword"],
            "crawl_time": video["crawl_time"],
            "match_score": 0.0
        })
    
    # 批量存入向量库
    COLLECTION.add(
        ids=ids,
        documents=documents,
        embeddings=embeddings,
        metadatas=metadatas
    )
    print(f"✅ 成功存入 {len(video_list)} 个视频到向量库")

# ====================== 3. 千问API功能 ======================
def call_qianwen_api(prompt: str, system_prompt: str = None, temperature: float = 0.3) -> str:
    """调用千问API"""
    try:
        messages = []
        
        if system_prompt:
            messages.append({"role": "system", "content": system_prompt})
        
        messages.append({"role": "user", "content": prompt})
        
        completion = QIANWEN_CLIENT.chat.completions.create(
            model=MODEL_NAME,
            messages=messages,
            temperature=temperature,
            max_tokens=800
        )
        
        return completion.choices[0].message.content.strip()
    except Exception as e:
        return f"❌ API调用失败: {str(e)}"

# ====================== 4. 智能查询理解 ======================
def understand_user_query(query: str) -> Dict[str, Any]:
    """使用千问API理解用户查询意图"""
    system_prompt = """你是一个B站视频推荐专家，请分析用户的查询意图。
    你需要识别：
    1. 主要主题/关键词
    2. 内容类型偏好（教程、娱乐、评测等）
    3. 视频特征（时长、风格、质量等）
    4. UP主偏好
    
    返回JSON格式。"""
    
    prompt = f"""请分析以下用户查询的意图：

用户查询: "{query}"

请返回JSON格式的分析结果，包括以下字段：
- main_topics: 主要主题列表
- content_types: 内容类型偏好列表
- video_features: 视频特征列表
- up_preferences: UP主偏好列表
- query_type: 查询类型（如"学习教程"、"娱乐放松"、"产品评测"等）"""

    response = call_qianwen_api(prompt, system_prompt)
    
    try:
        # 尝试从响应中提取JSON
        if "```json" in response:
            json_str = response.split("```json")[1].split("```")[0]
        elif "```" in response:
            json_str = response.split("```")[1].split("```")[0]
        else:
            json_str = response
        
        analysis = json.loads(json_str)
    except:
        # 如果解析失败，返回默认分析
        analysis = {
            "main_topics": [query],
            "content_types": ["综合"],
            "video_features": [],
            "up_preferences": [],
            "query_type": "综合搜索"
        }
    
    return analysis

# ====================== 5. 智能检索 + AI推荐 ======================
def smart_video_agent(user_query: str, top_k: int = 5, use_ai: bool = True) -> Dict[str, Any]:
    """智能视频Agent，支持AI增强推荐"""
    print(f"\n🤖 智能Agent正在处理查询: {user_query}")
    
    # 1. 查询理解
    query_analysis = understand_user_query(user_query) if use_ai else None
    
    # 2. 查询优化
    if use_ai and query_analysis:
        # 使用AI优化查询
        system_prompt = """你是一个视频搜索专家，请优化用户的查询，使其更适合视频检索。"""
        
        optimization_prompt = f"""原始查询: {user_query}
查询分析: {json.dumps(query_analysis, ensure_ascii=False, indent=2)}

请生成3个优化的搜索查询，用"|"分隔:"""
        
        optimized_queries = call_qianwen_api(optimization_prompt, system_prompt)
        queries = [q.strip() for q in optimized_queries.split("|") if q.strip()]
        queries = queries[:3]  # 取前3个
    else:
        queries = [user_query]
    
    print(f"🔍 优化后的查询: {queries}")
    
    # 3. 向量检索
    all_results = []
    seen_bvids = set()
    
    for query in queries:
        query_vec = EMBED_MODEL.encode(query).tolist()
        
        results = COLLECTION.query(
            query_embeddings=[query_vec],
            n_results=top_k * 2,  # 多取一些用于去重
            include=["metadatas", "distances", "documents"]
        )
        
        for i, meta in enumerate(results["metadatas"][0]):
            bvid = meta.get("bvid", "")
            if bvid in seen_bvids:
                continue
                
            # 计算匹配度
            distance = results["distances"][0][i]
            match_score = round(100 - distance * 100, 1)
            
            result = {
                "bvid": bvid,
                "title": meta.get("title", ""),
                "author": meta.get("author", ""),
                "url": meta.get("url", ""),
                "play_count": meta.get("play_count", 0),
                "publish_time": meta.get("publish_time", ""),
                "tags": meta.get("tags", "").split(",") if meta.get("tags") else [],
                "description": meta.get("description", ""),
                "match_score": match_score,
                "raw_distance": distance
            }
            
            all_results.append(result)
            seen_bvids.add(bvid)
    
    # 4. 排序和去重
    all_results.sort(key=lambda x: x["match_score"], reverse=True)
    final_results = all_results[:top_k]
    
    # 5. AI推荐理由生成
    if use_ai and final_results:
        recommendations = []
        
        for i, result in enumerate(final_results[:3]):  # 只为前3个生成推荐理由
            system_prompt = """你是一个视频推荐助手，请为用户生成个性化的推荐理由。"""
            
            prompt = f"""用户查询: {user_query}
查询分析: {json.dumps(query_analysis, ensure_ascii=False, indent=2) if query_analysis else "无"}

视频信息:
- 标题: {result['title']}
- UP主: {result['author']}
- 标签: {', '.join(result['tags'])}
- 描述: {result['description']}
- 发布时间: {result['publish_time']}
- 匹配度: {result['match_score']}%

请生成1-2句个性化的推荐理由，解释为什么这个视频适合用户:"""
            
            reason = call_qianwen_api(prompt, system_prompt, temperature=0.7)
            result["recommendation_reason"] = reason
            
            recommendations.append({
                "rank": i + 1,
                "title": result["title"],
                "author": result["author"],
                "url": result["url"],
                "match_score": result["match_score"],
                "reason": reason
            })
        
        # 总结推荐
        summary_prompt = f"""基于以下查询和推荐结果，生成一个简短的总结：

用户查询: {user_query}
推荐视频数量: {len(final_results)}
查询类型: {query_analysis.get('query_type', '综合搜索') if query_analysis else '综合搜索'}

请用1-2句话总结推荐内容:"""
        
        summary = call_qianwen_api(summary_prompt, "你是一个视频推荐总结专家。")
    else:
        summary = f"为您找到了 {len(final_results)} 个相关视频。"
    
    return {
        "query": user_query,
        "query_analysis": query_analysis,
        "optimized_queries": queries,
        "results": final_results,
        "recommendations": recommendations if 'recommendations' in locals() else [],
        "summary": summary,
        "total_found": len(final_results)
    }

# ====================== 6. 历史记录管理 ======================
def save_search_history(query: str, result_count: int):
    """保存搜索历史"""
    try:
        if os.path.exists(HISTORY_FILE):
            with open(HISTORY_FILE, 'r', encoding='utf-8') as f:
                history = json.load(f)
        else:
            history = []
        
        history.append({
            "timestamp": datetime.now().isoformat(),
            "query": query,
            "result_count": result_count
        })
        
        # 只保留最近50条记录
        if len(history) > 50:
            history = history[-50:]
        
        with open(HISTORY_FILE, 'w', encoding='utf-8') as f:
            json.dump(history, f, ensure_ascii=False, indent=2)
            
    except Exception as e:
        print(f"⚠️ 保存历史记录失败: {e}")

def get_search_history(limit: int = 10) -> List[Dict]:
    """获取搜索历史"""
    try:
        if os.path.exists(HISTORY_FILE):
            with open(HISTORY_FILE, 'r', encoding='utf-8') as f:
                history = json.load(f)
            return history[-limit:]
    except:
        pass
    return []

# ====================== 7. Gradio界面组件 ======================
def create_gradio_interface():
    """创建Gradio界面"""
    
    with gr.Blocks(title="B站智能视频推荐系统", theme=gr.themes.Soft()) as demo:
        gr.Markdown("""
        # 🎬 B站智能视频推荐系统
        基于千问API + 向量检索的智能视频发现工具
        """)
        
        # 标签页1: 爬取视频
        with gr.Tab("📥 爬取视频"):
            with gr.Row():
                with gr.Column(scale=2):
                    crawl_keyword = gr.Textbox(
                        label="搜索关键词",
                        placeholder="输入要爬取的B站视频关键词...",
                        lines=1
                    )
                    crawl_count = gr.Slider(
                        label="爬取数量",
                        minimum=5,
                        maximum=50,
                        value=20,
                        step=5
                    )
                    crawl_btn = gr.Button("开始爬取", variant="primary")
                
                with gr.Column(scale=3):
                    crawl_output = gr.Textbox(
                        label="爬取结果",
                        lines=10
                    )
                    crawl_stats = gr.Textbox(
                        label="统计信息",
                        lines=2
                    )
            
            def crawl_videos_func(keyword, count):
                if not keyword:
                    return "请输入关键词", "等待爬取..."
                
                start_time = time.time()
                videos = crawl_bilibili_videos(keyword, int(count))
                
                if videos:
                    save_to_vector_db(videos)
                    
                    # 统计信息
                    stats = []
                    for video in videos:
                        stats.append(f"• {video['title']} (播放: {video['play_count']})")
                    
                    output = "\n".join(stats[:10])
                    if len(stats) > 10:
                        output += f"\n... 还有 {len(stats)-10} 个视频"
                    
                    time_used = time.time() - start_time
                    stats_info = f"✅ 爬取完成 | 耗时: {time_used:.1f}s | 视频数: {len(videos)}"
                    
                    return output, stats_info
                else:
                    return "❌ 爬取失败或没有找到视频", "爬取失败"
            
            crawl_btn.click(
                fn=crawl_videos_func,
                inputs=[crawl_keyword, crawl_count],
                outputs=[crawl_output, crawl_stats]
            )
        
        # 标签页2: 智能推荐
        with gr.Tab("🔍 智能推荐"):
            with gr.Row():
                with gr.Column(scale=2):
                    query_input = gr.Textbox(
                        label="你想看什么视频？",
                        placeholder="例如：Python入门教程、科技产品评测、搞笑剪辑...",
                        lines=2
                    )
                    
                    with gr.Row():
                        top_k_slider = gr.Slider(
                            label="推荐数量",
                            minimum=1,
                            maximum=10,
                            value=5,
                            step=1
                        )
                        use_ai_check = gr.Checkbox(
                            label="启用AI智能推荐",
                            value=True
                        )
                    
                    search_btn = gr.Button("🚀 智能搜索", variant="primary", scale=1)
                    
                    with gr.Accordion("📊 查询分析", open=False):
                        query_analysis_output = gr.JSON(
                            label="查询意图分析"
                        )
                
                with gr.Column(scale=3):
                    search_summary = gr.Textbox(
                        label="推荐总结",
                        lines=2
                    )
                    
                    with gr.Tabs():
                        with gr.TabItem("📋 列表视图"):
                            results_df = gr.Dataframe(
                                headers=["排名", "标题", "UP主", "匹配度", "播放量", "发布时间"],
                                wrap=True
                            )
                        
                        with gr.TabItem("🎯 详细推荐"):
                            recommendations_html = gr.HTML(
                                label="AI推荐详情"
                            )
                    
                    with gr.Accordion("🔗 快速访问", open=False):
                        urls_output = gr.Textbox(
                            label="视频链接",
                            lines=5
                        )
            
            def search_videos_func(query, top_k, use_ai):
                if not query:
                    return {}, "请输入查询", pd.DataFrame(), "", "", ""
                
                start_time = time.time()
                
                # 智能搜索
                result = smart_video_agent(query, int(top_k), use_ai)
                
                # 保存历史
                save_search_history(query, len(result["results"]))
                
                # 准备数据
                time_used = time.time() - start_time
                summary = f"{result['summary']} | 耗时: {time_used:.1f}s"
                
                # 数据表格
                df_data = []
                for i, video in enumerate(result["results"], 1):
                    df_data.append([
                        i,
                        video["title"][:50] + ("..." if len(video["title"]) > 50 else ""),
                        video["author"],
                        f"{video['match_score']}%",
                        f"{video['play_count']:,}" if video.get('play_count') else "未知",
                        video.get("publish_time", "未知")
                    ])
                
                df = pd.DataFrame(df_data, columns=["排名", "标题", "UP主", "匹配度", "播放量", "发布时间"])
                
                # 推荐HTML
                html_content = ""
                if result.get("recommendations"):
                    for rec in result["recommendations"]:
                        html_content += f"""
                        <div style="border: 1px solid #ddd; padding: 15px; margin: 10px 0; border-radius: 8px;">
                            <h3>🎯 推荐 #{rec['rank']} (匹配度: {rec['match_score']}%)</h3>
                            <p><strong>标题:</strong> {rec['title']}</p>
                            <p><strong>UP主:</strong> {rec['author']}</p>
                            <p><strong>推荐理由:</strong> {rec['reason']}</p>
                            <p><a href="{rec['url']}" target="_blank">点击观看</a></p>
                        </div>
                        """
                else:
                    html_content = "<p>AI推荐未启用或无推荐结果</p>"
                
                # 链接列表
                urls = "\n".join([f"{i+1}. {v['title'][:30]}...\n   {v['url']}" 
                                 for i, v in enumerate(result["results"][:3])])
                
                return result.get("query_analysis", {}), summary, df, html_content, urls, query
            
            search_btn.click(
                fn=search_videos_func,
                inputs=[query_input, top_k_slider, use_ai_check],
                outputs=[query_analysis_output, search_summary, results_df, recommendations_html, urls_output, query_input]
            )
        
        # 标签页3: 历史记录
        with gr.Tab("📜 搜索历史"):
            with gr.Row():
                with gr.Column(scale=2):
                    refresh_btn = gr.Button("🔄 刷新历史", variant="secondary")
                
                with gr.Column(scale=3):
                    history_df = gr.Dataframe(
                        headers=["时间", "查询", "结果数"]
                    )
            
            def load_history():
                history = get_search_history(20)
                data = []
                for item in reversed(history):  # 倒序显示，最新的在前面
                    time_str = item["timestamp"][11:19]  # 只显示时间
                    data.append([time_str, item["query"], item["result_count"]])
                return pd.DataFrame(data)
            
            refresh_btn.click(fn=load_history, outputs=[history_df])
            demo.load(fn=load_history, outputs=[history_df])
        
        # 标签页4: 系统信息
        with gr.Tab("ℹ️ 系统信息"):
            gr.Markdown("""
            ### 🎯 系统功能
            
            **核心功能:**
            1. **智能爬取**: 自动爬取B站视频并向量化存储
            2. **语义检索**: 基于向量相似度的精准匹配
            3. **AI增强**: 使用千问API进行查询理解和推荐
            4. **智能推荐**: 个性化推荐理由生成
            
            **技术栈:**
            - **向量数据库**: Chroma (本地持久化)
            - **嵌入模型**: all-MiniLM-L6-v2 (轻量级)
            - **大语言模型**: 通义千问 (Qwen-Plus)
            - **Web框架**: Gradio
            - **爬虫**: Requests + B站开放API
            
            **使用说明:**
            1. 在"爬取视频"标签页输入关键词，获取视频数据
            2. 在"智能推荐"标签页输入你想看的内容类型
            3. 系统会基于语义匹配和AI分析推荐最相关的视频
            4. 查看"搜索历史"了解过往搜索记录
            
            **注意事项:**
            - 请合理使用爬虫功能，避免对B站服务器造成压力
            - API调用需要设置 DASHSCOPE_API_KEY 环境变量
            - 首次使用前需要先爬取一些视频数据
            """)
            
            # 向量库统计
            stats_text = gr.Textbox(
                label="向量库统计",
                value="正在加载..."
            )
            
            def get_stats():
                try:
                    count = COLLECTION.count()
                    return f"📊 向量库状态\n当前视频数量: {count} 个"
                except:
                    return "无法获取向量库信息"
            
            demo.load(fn=get_stats, outputs=[stats_text])
    
    return demo

# ====================== 8. 辅助功能 ======================
def get_vector_db_stats():
    """获取向量库统计信息"""
    try:
        count = COLLECTION.count()
        
        # 获取一些示例数据
        sample = COLLECTION.peek(limit=3)
        sample_titles = [item.get("title", "未知") for item in sample.get("metadatas", [])]
        
        stats = {
            "total_videos": count,
            "sample_titles": sample_titles[:3]
        }
        
        return stats
    except Exception as e:
        return {"error": str(e)}

# ====================== 9. 主程序 ======================
if __name__ == "__main__":
    # 检查API密钥
    if not os.getenv("DASHSCOPE_API_KEY"):
        print("⚠️ 警告: 未设置 DASHSCOPE_API_KEY 环境变量")
        print("请运行: export DASHSCOPE_API_KEY=sk-xxx")
        print("或创建 .env 文件")
    
    # 显示初始信息
    print("="*60)
    print("🚀 B站智能视频推荐系统 启动中...")
    print("="*60)
    
    # 显示向量库状态
    stats = get_vector_db_stats()
    if "total_videos" in stats:
        print(f"📊 当前向量库有 {stats['total_videos']} 个视频")
    else:
        print("📊 向量库为空，请先爬取一些视频")
    
    # 创建并启动Gradio界面
    demo = create_gradio_interface()
    
    print("\n🌐 启动Web界面...")
    print("📱 访问: http://localhost:7860")
    print("\n按 Ctrl+C 退出程序")
    
    try:
        demo.launch(
            server_name="0.0.0.0",
            server_port=7860,
            share=False,
            debug=False
        )
    except KeyboardInterrupt:
        print("\n👋 程序已退出")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: /home/dake/Documents/Project/Life/data/datasets/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_89636/4256208726.py:414: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="B站智能视频推荐系统", theme=gr.themes.Soft()) as demo:


🚀 B站智能视频推荐系统 启动中...
📊 当前向量库有 20 个视频

🌐 启动Web界面...
📱 访问: http://localhost:7860

按 Ctrl+C 退出程序
* Running on local URL:  http://0.0.0.0:7860
* To create a public link, set `share=True` in `launch()`.


🔍 开始爬取关键词: 小鸡小鸡
📄 第1页爬取完成，已获取20个视频
✅ 爬取完成，共获取 20 个视频
✅ 成功存入 20 个视频到向量库

🤖 智能Agent正在处理查询: 小鸡产后护理
🔍 优化后的查询: ["❌ API调用失败: Error code: 404 - {'request_id': '692d7ff4-fdf6-92c8-9e91-6cd06cdee54e', 'code': 'Not Found', 'message': 'Not support'}"]


In [6]:
#!/usr/bin/env python3
"""
B站智能视频推荐系统
重构版本：遵循Gradio官方文档，优化架构，解决API 404错误
"""

import os
import sys
import time
import json
import hashlib
import chromadb
import gradio as gr
import requests
import pandas as pd
from typing import List, Dict, Any, Optional, Tuple
from datetime import datetime
from dataclasses import dataclass, asdict, field
from enum import Enum
import logging
from sentence_transformers import SentenceTransformer
from openai import OpenAI

# 配置日志
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# ====================== 数据模型 ======================
@dataclass
class VideoInfo:
    """视频信息数据类"""
    bvid: str
    title: str
    author: str
    url: str
    play_count: int = 0
    duration: str = ""
    publish_time: str = ""
    tags: List[str] = field(default_factory=list)
    description: str = ""
    search_keyword: str = ""
    crawl_time: str = ""
    match_score: float = 0.0
    
    def to_dict(self) -> Dict[str, Any]:
        """转换为字典"""
        return asdict(self)
    
    @classmethod
    def from_dict(cls, data: Dict[str, Any]) -> 'VideoInfo':
        """从字典创建"""
        return cls(**data)

@dataclass
class SearchHistory:
    """搜索历史数据类"""
    id: str
    query: str
    timestamp: str
    result_count: int
    results: List[Dict[str, Any]] = field(default_factory=list)
    
    def to_dict(self) -> Dict[str, Any]:
        """转换为字典"""
        return asdict(self)

# ====================== 数据库管理类 ======================
class DatabaseManager:
    """数据库管理器 - 管理Chroma向量库和元数据"""
    
    def __init__(self, persist_dir: str = "./video_db"):
        """初始化数据库管理器"""
        self.persist_dir = persist_dir
        self.client = chromadb.PersistentClient(path=persist_dir)
        
        # 视频向量库
        self.video_collection = self.client.get_or_create_collection(
            name="bilibili_videos",
            metadata={"hnsw:space": "cosine"}
        )
        
        # 搜索历史库
        self.history_collection = self.client.get_or_create_collection(
            name="search_history",
            metadata={"hnsw:space": "cosine"}
        )
        
        logger.info(f"✅ 数据库初始化完成，路径: {persist_dir}")
    
    def add_videos(self, videos: List[VideoInfo], embedder) -> int:
        """添加视频到向量库"""
        if not videos:
            return 0
        
        ids = []
        documents = []
        embeddings = []
        metadatas = []
        
        for video in videos:
            # 创建文档内容
            content = self._create_video_document(video)
            
            # 生成向量
            embedding = embedder.encode(content).tolist()
            
            # 生成唯一ID
            video_id = f"{video.bvid}_{hashlib.md5(content.encode()).hexdigest()[:8]}"
            
            ids.append(video_id)
            documents.append(content)
            embeddings.append(embedding)
            metadatas.append(video.to_dict())
        
        # 批量添加
        self.video_collection.add(
            ids=ids,
            documents=documents,
            embeddings=embeddings,
            metadatas=metadatas
        )
        
        logger.info(f"✅ 成功添加 {len(videos)} 个视频到向量库")
        return len(videos)
    
    def search_videos(self, query_embedding: List[float], top_k: int = 5) -> List[VideoInfo]:
        """搜索相似视频"""
        try:
            results = self.video_collection.query(
                query_embeddings=[query_embedding],
                n_results=top_k,
                include=["metadatas", "distances"]
            )
            
            videos = []
            for i, meta in enumerate(results["metadatas"][0]):
                # 计算匹配分数
                distance = results["distances"][0][i]
                match_score = round(100 - distance * 100, 1)
                
                # 创建视频对象
                video = VideoInfo.from_dict(meta)
                video.match_score = match_score
                videos.append(video)
            
            return videos
        except Exception as e:
            logger.error(f"搜索视频失败: {e}")
            return []
    
    def get_video_count(self) -> int:
        """获取视频数量"""
        try:
            return self.video_collection.count()
        except:
            return 0
    
    def get_sample_videos(self, limit: int = 5) -> List[VideoInfo]:
        """获取示例视频"""
        try:
            results = self.video_collection.peek(limit=limit)
            videos = []
            for meta in results.get("metadatas", []):
                videos.append(VideoInfo.from_dict(meta))
            return videos
        except:
            return []
    
    def add_search_history(self, query: str, results: List[VideoInfo]) -> str:
        """添加搜索历史"""
        history_id = f"history_{hashlib.md5(f'{query}_{datetime.now().isoformat()}'.encode()).hexdigest()[:12]}"
        
        history = SearchHistory(
            id=history_id,
            query=query,
            timestamp=datetime.now().isoformat(),
            result_count=len(results),
            results=[v.to_dict() for v in results]
        )
        
        # 向量化查询用于搜索
        query_embedding = [0.0] * 384  # 简单占位向量
        
        self.history_collection.add(
            ids=[history_id],
            documents=[query],
            embeddings=[query_embedding],
            metadatas=[history.to_dict()]
        )
        
        return history_id
    
    def get_search_history(self, limit: int = 10) -> List[SearchHistory]:
        """获取搜索历史"""
        try:
            results = self.history_collection.peek(limit=limit)
            history_list = []
            
            for meta in results.get("metadatas", []):
                history = SearchHistory.from_dict(meta)
                history_list.append(history)
            
            return history_list
        except Exception as e:
            logger.error(f"获取搜索历史失败: {e}")
            return []
    
    def _create_video_document(self, video: VideoInfo) -> str:
        """创建视频文档文本用于向量化"""
        parts = [
            f"标题: {video.title}",
            f"作者: {video.author}",
            f"标签: {', '.join(video.tags)}",
            f"描述: {video.description}",
            f"关键词: {video.search_keyword}"
        ]
        return " | ".join(parts)

# ====================== B站爬虫类 ======================
class BilibiliCrawler:
    """B站视频爬虫"""
    
    def __init__(self):
        self.headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
            "Referer": "https://www.bilibili.com/"
        }
        self.base_url = "https://api.bilibili.com/x/web-interface/search/all/v2"
    
    def crawl_videos(self, keyword: str, max_count: int = 20) -> Tuple[List[VideoInfo], str]:
        """爬取B站视频"""
        videos = []
        page = 1
        error_msg = ""
        
        logger.info(f"开始爬取关键词: {keyword}")
        
        while len(videos) < max_count:
            try:
                # 构造请求URL
                params = {
                    "keyword": keyword,
                    "page": page,
                    "page_size": 20
                }
                
                response = requests.get(
                    self.base_url, 
                    headers=self.headers, 
                    params=params,
                    timeout=10
                )
                response.raise_for_status()
                data = response.json()
                
                # 检查响应
                if data.get("code") != 0:
                    error_msg = f"API返回错误: {data.get('message', '未知错误')}"
                    logger.error(error_msg)
                    break
                
                # 提取视频数据
                video_items = self._extract_video_items(data)
                
                if not video_items:
                    logger.info("没有更多视频数据")
                    break
                
                for item in video_items:
                    if len(videos) >= max_count:
                        break
                    
                    video = self._parse_video_item(item, keyword)
                    if video:
                        videos.append(video)
                
                page += 1
                time.sleep(1.5)  # 防止请求过快
                
            except requests.exceptions.Timeout:
                error_msg = "请求超时"
                logger.error("请求超时")
                break
            except requests.exceptions.RequestException as e:
                error_msg = f"网络请求失败: {str(e)}"
                logger.error(f"网络请求失败: {e}")
                break
            except Exception as e:
                error_msg = f"解析数据失败: {str(e)}"
                logger.error(f"解析数据失败: {e}")
                break
        
        logger.info(f"爬取完成，获取 {len(videos)} 个视频")
        return videos, error_msg
    
    def _extract_video_items(self, data: Dict[str, Any]) -> List[Dict[str, Any]]:
        """从API响应中提取视频项"""
        video_items = []
        
        try:
            # 查找视频类型的结果
            for result in data.get("data", {}).get("result", []):
                if result.get("result_type") == "video":
                    video_items = result.get("data", [])
                    break
            
            # 如果没有找到，尝试其他结构
            if not video_items and "data" in data:
                # 尝试直接获取
                video_items = data["data"].get("result", [])
        
        except Exception as e:
            logger.error(f"提取视频项失败: {e}")
        
        return video_items
    
    def _parse_video_item(self, item: Dict[str, Any], keyword: str) -> Optional[VideoInfo]:
        """解析单个视频项"""
        try:
            # 提取基本信息
            bvid = item.get("bvid", "")
            if not bvid:
                return None
            
            title = item.get("title", "").replace("<em class=\"keyword\">", "").replace("</em>", "")
            author = item.get("author", "")
            play_count = item.get("play", 0)
            duration = item.get("duration", "")
            
            # 处理标签
            tags_str = item.get("tag", "")
            if tags_str:
                tags = [tag.strip() for tag in tags_str.split(",") if tag.strip()]
            else:
                tags = []
            
            # 处理描述
            description = item.get("description", "")[:200]
            
            # 处理发布时间
            pubdate = item.get("pubdate", 0)
            if pubdate:
                publish_time = datetime.fromtimestamp(pubdate).strftime("%Y-%m-%d %H:%M")
            else:
                publish_time = datetime.now().strftime("%Y-%m-%d %H:%M")
            
            return VideoInfo(
                bvid=bvid,
                title=title,
                author=author,
                url=f"https://www.bilibili.com/video/{bvid}",
                play_count=play_count,
                duration=duration,
                publish_time=publish_time,
                tags=tags,
                description=description,
                search_keyword=keyword,
                crawl_time=datetime.now().isoformat()
            )
        
        except Exception as e:
            logger.error(f"解析视频项失败: {e}")
            return None

# ====================== 千问API客户端 ======================
class QwenAPIClient:
    """千问API客户端 - 使用正确的API调用方式"""
    
    def __init__(self, api_key: str = None):
        self.api_key = api_key or os.getenv("DASHSCOPE_API_KEY", "")
        
        if not self.api_key:
            logger.warning("未设置DASHSCOPE_API_KEY，部分AI功能将不可用")
        
        # 初始化OpenAI客户端
        self.client = OpenAI(
            api_key=self.api_key,
            base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
        )
        
        self.model = "qwen-plus"
        logger.info(f"✅ 千问API客户端初始化完成 (模型: {self.model})")
    
    def is_available(self) -> bool:
        """检查API是否可用"""
        return bool(self.api_key)
    
    def chat_completion(self, messages: List[Dict[str, str]], **kwargs) -> str:
        """聊天补全"""
        if not self.is_available():
            return "⚠️ 未配置API密钥，请设置DASHSCOPE_API_KEY环境变量"
        
        try:
            completion = self.client.chat.completions.create(
                model=self.model,
                messages=messages,
                **kwargs
            )
            
            return completion.choices[0].message.content
            
        except Exception as e:
            error_msg = f"API调用失败: {str(e)}"
            logger.error(error_msg)
            return f"❌ {error_msg}"
    
    def optimize_query(self, query: str) -> List[str]:
        """优化搜索查询"""
        if not self.is_available():
            return [query]
        
        system_prompt = "你是一个搜索引擎优化专家，请优化用户的搜索查询以提高检索效果。"
        user_prompt = f"请将以下查询优化为3个更适合视频检索的版本，用'|'分隔:\n\n原始查询: {query}\n\n优化后的查询:"
        
        response = self.chat_completion([
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ], temperature=0.3)
        
        # 解析响应
        if "|" in response:
            queries = [q.strip() for q in response.split("|") if q.strip()]
        else:
            queries = [response.strip()]
        
        return queries[:3]  # 最多返回3个
    
    def generate_recommendation(self, query: str, video: VideoInfo) -> str:
        """生成视频推荐理由"""
        if not self.is_available():
            return f"视频《{video.title}》与您的查询相关"
        
        system_prompt = "你是一个视频推荐助手，请为用户生成个性化的推荐理由。"
        
        user_prompt = f"""用户查询: {query}

视频信息:
- 标题: {video.title}
- UP主: {video.author}
- 标签: {', '.join(video.tags[:5])}
- 描述: {video.description[:100]}
- 发布时间: {video.publish_time}
- 播放量: {video.play_count}
- 匹配度: {video.match_score}%

请用1-2句话说明为什么推荐这个视频:"""
        
        return self.chat_completion([
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ], temperature=0.7)

# ====================== 智能视频推荐系统 ======================
class SmartVideoRecommender:
    """智能视频推荐系统"""
    import os 
    minilm = str(os.path.join("/run/media/dake/Software/Projects/Life/data/datasets/all-MiniLM-L6-v2"))
    def __init__(self, embed_model_path: str = minilm):
        # 初始化组件
        self.embedder = SentenceTransformer(embed_model_path)
        self.db = DatabaseManager()
        self.crawler = BilibiliCrawler()
        self.qwen_client = QwenAPIClient()
        
        logger.info("✅ 智能视频推荐系统初始化完成")
    
    def crawl_and_store(self, keyword: str, max_count: int = 20) -> Tuple[str, str]:
        """爬取并存储视频"""
        start_time = time.time()
        
        # 爬取视频
        videos, error_msg = self.crawler.crawl_videos(keyword, max_count)
        
        if error_msg:
            return f"❌ 爬取失败: {error_msg}", ""
        
        if not videos:
            return "⚠️ 没有找到相关视频", ""
        
        # 存储到数据库
        count = self.db.add_videos(videos, self.embedder)
        
        time_used = time.time() - start_time
        
        # 生成统计信息
        stats = f"✅ 爬取完成\n\n"
        stats += f"• 关键词: {keyword}\n"
        stats += f"• 获取视频: {len(videos)} 个\n"
        stats += f"• 存储成功: {count} 个\n"
        stats += f"• 耗时: {time_used:.1f}秒\n"
        stats += f"• 当前库中视频总数: {self.db.get_video_count()} 个\n\n"
        
        # 显示前3个视频
        sample = ""
        for i, video in enumerate(videos[:3], 1):
            sample += f"{i}. {video.title} (播放: {video.play_count:,})\n"
        
        return stats, sample
    
    def search_videos(self, query: str, top_k: int = 5, use_ai: bool = True) -> Dict[str, Any]:
        """搜索视频"""
        start_time = time.time()
        
        logger.info(f"搜索查询: {query}")
        
        # 查询优化
        if use_ai and self.qwen_client.is_available():
            optimized_queries = self.qwen_client.optimize_query(query)
            if optimized_queries and optimized_queries != [query]:
                logger.info(f"优化后的查询: {optimized_queries}")
                search_query = optimized_queries[0]  # 使用第一个优化查询
            else:
                search_query = query
        else:
            search_query = query
            optimized_queries = [query]
        
        # 生成查询向量
        query_embedding = self.embedder.encode(search_query).tolist()
        
        # 搜索数据库
        results = self.db.search_videos(query_embedding, top_k)
        
        # 生成推荐理由
        recommendations = []
        if use_ai and self.qwen_client.is_available():
            for video in results[:3]:  # 只为前3个生成推荐理由
                try:
                    reason = self.qwen_client.generate_recommendation(query, video)
                    recommendations.append({
                        "title": video.title,
                        "author": video.author,
                        "url": video.url,
                        "match_score": video.match_score,
                        "reason": reason
                    })
                except Exception as e:
                    logger.error(f"生成推荐理由失败: {e}")
                    recommendations.append({
                        "title": video.title,
                        "author": video.author,
                        "url": video.url,
                        "match_score": video.match_score,
                        "reason": f"视频《{video.title}》与您的查询相关"
                    })
        
        # 保存搜索历史
        if results:
            self.db.add_search_history(query, results)
        
        time_used = time.time() - start_time
        
        return {
            "query": query,
            "optimized_queries": optimized_queries,
            "results": results,
            "recommendations": recommendations,
            "search_time": time_used,
            "total_found": len(results)
        }
    
    def get_search_history(self, limit: int = 10) -> List[Dict]:
        """获取搜索历史"""
        history = self.db.get_search_history(limit)
        
        formatted = []
        for h in history:
            formatted.append({
                "时间": h.timestamp[11:19],  # 只显示时间
                "查询": h.query[:30] + ("..." if len(h.query) > 30 else ""),
                "结果数": h.result_count
            })
        
        return formatted
    
    def get_system_stats(self) -> Dict[str, Any]:
        """获取系统统计信息"""
        video_count = self.db.get_video_count()
        sample_videos = self.db.get_sample_videos(3)
        
        stats = {
            "视频总数": video_count,
            "样本视频": [v.title for v in sample_videos],
            "API可用": self.qwen_client.is_available(),
            "嵌入模型": self.embedder.get_sentence_embedding_dimension()
        }
        
        return stats

# ====================== Gradio界面 ======================
class VideoAppUI:
    """视频应用UI"""
    
    def __init__(self, recommender: SmartVideoRecommender):
        self.recommender = recommender
        
    def create_interface(self):
        """创建Gradio界面"""
        with gr.Blocks(title="B站智能视频推荐系统", theme=gr.themes.Soft()) as demo:
            gr.Markdown("# 🎬 B站智能视频推荐系统")
            gr.Markdown("智能爬取、搜索和推荐B站视频")
            
            # 主标签页
            with gr.Tab("🔍 搜索与推荐"):
                with gr.Row():
                    with gr.Column(scale=2):
                        # 搜索区域
                        search_input = gr.Textbox(
                            label="你想看什么视频？",
                            placeholder="输入关键词，如：Python教程、科技评测、美食制作...",
                            lines=2
                        )
                        
                        with gr.Row():
                            top_k_slider = gr.Slider(
                                label="返回结果数",
                                minimum=1,
                                maximum=10,
                                value=5,
                                step=1
                            )
                            use_ai_check = gr.Checkbox(
                                label="启用AI优化",
                                value=True
                            )
                        
                        search_btn = gr.Button("🔍 智能搜索", variant="primary")
                        
                        # 爬取区域
                        with gr.Group():
                            gr.Markdown("### 📥 爬取新视频")
                            with gr.Row():
                                crawl_input = gr.Textbox(
                                    label="爬取关键词",
                                    placeholder="输入要爬取的B站视频关键词...",
                                    scale=3
                                )
                                crawl_count = gr.Number(
                                    label="数量",
                                    value=10,
                                    minimum=5,
                                    maximum=30,
                                    precision=0,
                                    scale=1
                                )
                            crawl_btn = gr.Button("🚀 开始爬取", variant="secondary")
                    
                    with gr.Column(scale=3):
                        # 搜索结果
                        search_output = gr.Textbox(
                            label="搜索结果",
                            lines=8,
                            interactive=False
                        )
                        
                        # 视频列表
                        results_table = gr.Dataframe(
                            headers=["排名", "标题", "UP主", "匹配度", "播放量", "发布时间"],
                            interactive=False,
                            wrap=True
                        )
                        
                        # 推荐理由
                        recommendations_html = gr.HTML(
                            label="AI推荐理由"
                        )
                
                # 功能区域
                with gr.Row():
                    with gr.Column():
                        # 快速链接
                        with gr.Accordion("🔗 视频链接", open=False):
                            video_links = gr.Textbox(
                                label="视频链接",
                                lines=3,
                                interactive=False
                            )
                    
                    with gr.Column():
                        # 搜索历史
                        with gr.Accordion("📜 搜索历史", open=False):
                            history_table = gr.Dataframe(
                                headers=["时间", "查询", "结果数"],
                                interactive=False
                            )
                            refresh_btn = gr.Button("🔄 刷新历史", size="sm")
            
            # 系统信息标签页
            with gr.Tab("ℹ️ 系统信息"):
                with gr.Row():
                    with gr.Column(scale=2):
                        gr.Markdown("""
                        ### 🎯 系统功能
                        
                        **核心功能:**
                        1. **智能爬取**: 自动爬取B站视频并向量化存储
                        2. **语义检索**: 基于向量相似度的精准匹配
                        3. **AI优化**: 使用千问API优化搜索和生成推荐理由
                        
                        **技术栈:**
                        - 向量数据库: Chroma
                        - 嵌入模型: Sentence Transformers
                        - AI模型: 通义千问 (Qwen-Plus)
                        - Web框架: Gradio
                        
                        **使用说明:**
                        1. 在左侧输入关键词搜索视频
                        2. 如需新内容，可爬取B站视频入库
                        3. 启用AI优化可获得更好的搜索和推荐
                        """)
                    
                    with gr.Column(scale=1):
                        # 系统统计
                        stats_output = gr.Textbox(
                            label="📊 系统统计",
                            lines=8,
                            interactive=False
                        )
                        
                        refresh_stats_btn = gr.Button("🔄 刷新统计")
            
            # ====================== 事件处理 ======================
            
            def handle_search(query, top_k, use_ai):
                """处理搜索请求"""
                if not query:
                    return "请输入搜索内容", pd.DataFrame(), "", "", [], pd.DataFrame()
                
                result = self.recommender.search_videos(query, int(top_k), use_ai)
                
                # 搜索结果文本
                search_text = f"🔍 搜索: {result['query']}\n"
                search_text += f"📊 找到 {result['total_found']} 个相关视频\n"
                search_text += f"⏱️ 耗时: {result['search_time']:.2f}秒\n"
                
                if result.get('optimized_queries') and len(result['optimized_queries']) > 1:
                    search_text += f"🔄 优化查询: {' | '.join(result['optimized_queries'][1:])}\n"
                
                # 结果表格
                df_data = []
                for i, video in enumerate(result["results"], 1):
                    df_data.append([
                        i,
                        video.title[:40] + ("..." if len(video.title) > 40 else ""),
                        video.author[:15],
                        f"{video.match_score}%",
                        f"{video.play_count:,}",
                        video.publish_time
                    ])
                
                df = pd.DataFrame(df_data, columns=["排名", "标题", "UP主", "匹配度", "播放量", "发布时间"])
                
                # 推荐理由HTML
                html_content = ""
                if result.get("recommendations"):
                    for rec in result["recommendations"]:
                        html_content += f"""
                        <div style="border: 1px solid #e0e0e0; padding: 12px; margin: 8px 0; border-radius: 6px; background: #f9f9f9;">
                            <p><strong>🎯 {rec['title'][:30]}{'...' if len(rec['title']) > 30 else ''}</strong></p>
                            <p><small>UP: {rec['author']} | 匹配度: {rec['match_score']}%</small></p>
                            <p>{rec['reason']}</p>
                            <p><a href="{rec['url']}" target="_blank" style="color: #00a1d6;">📺 观看视频</a></p>
                        </div>
                        """
                else:
                    html_content = "<p>启用AI优化可获得个性化推荐理由</p>"
                
                # 视频链接
                links = "\n".join([f"{i+1}. {v.title[:30]}...\n   {v.url}" 
                                  for i, v in enumerate(result["results"][:3])])
                
                # 获取最新历史
                history = self.recommender.get_search_history(5)
                history_df = pd.DataFrame(history, columns=["时间", "查询", "结果数"])
                
                return search_text, df, html_content, links, history, history_df
            
            def handle_crawl(keyword, count):
                """处理爬取请求"""
                if not keyword:
                    return "请输入爬取关键词", ""
                
                stats, sample = self.recommender.crawl_and_store(keyword, int(count))
                return stats, sample
            
            def get_current_stats():
                """获取当前系统统计"""
                stats = self.recommender.get_system_stats()
                
                stats_text = f"📈 系统状态\n"
                stats_text += f"• 视频总数: {stats['视频总数']} 个\n"
                stats_text += f"• API状态: {'✅ 可用' if stats['API可用'] else '❌ 不可用'}\n"
                stats_text += f"• 向量维度: {stats['嵌入模型']}\n"
                
                if stats['样本视频']:
                    stats_text += f"\n📺 样本视频:\n"
                    for i, title in enumerate(stats['样本视频'], 1):
                        stats_text += f"  {i}. {title[:30]}...\n"
                
                return stats_text
            
            def get_search_history_df():
                """获取搜索历史DataFrame"""
                history = self.recommender.get_search_history(10)
                return pd.DataFrame(history, columns=["时间", "查询", "结果数"])
            
            # ====================== 绑定事件 ======================
            
            # 搜索按钮
            search_btn.click(
                fn=handle_search,
                inputs=[search_input, top_k_slider, use_ai_check],
                outputs=[search_output, results_table, recommendations_html, video_links, history_table]
            )
            
            # 爬取按钮
            crawl_btn.click(
                fn=handle_crawl,
                inputs=[crawl_input, crawl_count],
                outputs=[search_output, results_table]
            )
            
            # 刷新历史按钮
            refresh_btn.click(
                fn=get_search_history_df,
                outputs=[history_table]
            )
            
            # 刷新统计按钮
            refresh_stats_btn.click(
                fn=get_current_stats,
                outputs=[stats_output]
            )
            
            # 页面加载时初始化
            demo.load(get_current_stats, outputs=[stats_output])
            demo.load(get_search_history_df, outputs=[history_table])
        
        return demo

# ====================== 主程序 ======================
def main():
    """主程序"""
    print("="*60)
    print("🚀 B站智能视频推荐系统 启动中...")
    print("="*60)
    
    # 检查API密钥
    api_key = os.getenv("DASHSCOPE_API_KEY")
    if not api_key:
        print("⚠️ 注意: 未设置DASHSCOPE_API_KEY环境变量")
        print("   部分AI功能将受限，但基础搜索功能仍可用")
        print("   如需完整功能，请设置: export DASHSCOPE_API_KEY=sk-xxx")
    
    # 初始化推荐系统
    try:
        # 使用相对路径的模型
        embed_model_path = "all-MiniLM-L6-v2"
        
        recommender = SmartVideoRecommender(embed_model_path)
        app_ui = VideoAppUI(recommender)
        
        # 创建并启动界面
        demo = app_ui.create_interface()
        
        print("\n🌐 Web界面启动中...")
        print("📱 请访问: http://localhost:7860")
        print("\n按 Ctrl+C 退出程序")
        
        # 启动Gradio
        demo.launch(
            server_name="0.0.0.0",
            server_port=7860,
            share=False
        )
        
    except KeyboardInterrupt:
        print("\n👋 程序已退出")
    except Exception as e:
        print(f"\n❌ 启动失败: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

2026-04-06 11:28:18,877 - sentence_transformers.SentenceTransformer - INFO - Use pytorch device_name: cuda:0
2026-04-06 11:28:18,877 - sentence_transformers.SentenceTransformer - INFO - Load pretrained SentenceTransformer: all-MiniLM-L6-v2
'[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1000)' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/./modules.json
2026-04-06 11:28:18,892 - huggingface_hub.utils._http - WARNING - '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1000)' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].
2026-04-06 11:28:18,893 - huggingface_hub.utils._http - WARNING - Retrying in 1s [Retry 1/5].


🚀 B站智能视频推荐系统 启动中...


2026-04-06 11:28:19,894 - sentence_transformers.SentenceTransformer - WARNING - No sentence-transformers model found with name sentence-transformers/all-MiniLM-L6-v2. Creating a new one with mean pooling.
'[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1000)' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/adapter_config.json
2026-04-06 11:28:19,911 - huggingface_hub.utils._http - WARNING - '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1000)' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/adapter_config.json
Retrying in 1s [Retry 1/5].
2026-04-06 11:28:19,911 - huggingface_hub.utils._http - WARNING - Retrying in 1s [Retry 1/5].



❌ 启动失败: Cannot send a request, as the client has been closed.


Traceback (most recent call last):
  File "/tmp/ipykernel_89636/2741615885.py", line 863, in main
    recommender = SmartVideoRecommender(embed_model_path)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_89636/2741615885.py", line 463, in __init__
    self.embedder = SentenceTransformer(embed_model_path)
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/dake/.conda/envs/lang/lib/python3.12/site-packages/sentence_transformers/SentenceTransformer.py", line 339, in __init__
    modules = self._load_auto_model(
              ^^^^^^^^^^^^^^^^^^^^^^
  File "/home/dake/.conda/envs/lang/lib/python3.12/site-packages/sentence_transformers/SentenceTransformer.py", line 2130, in _load_auto_model
    transformer_model = Transformer(
                        ^^^^^^^^^^^^
  File "/home/dake/.conda/envs/lang/lib/python3.12/site-packages/sentence_transformers/models/Transformer.py", line 120, in __init__
    config, is_peft_model = self._load_config